# Module 03 — AI Agents
## Lesson 11 — Compare Agent Frameworks

Now that we have implemented the mechanics ourselves, we can evaluate a framework from a position of understanding rather than treating it as magic.

This lesson uses **LangGraph** as the main comparison because it is a low-level orchestration framework for stateful agents. We deliberately keep direct OpenAI Responses API calls rather than switching to a higher-level agent abstraction at the same time.


## The comparison target

Lesson 10 explicitly implemented:

- state;
- repeated model/tool execution;
- routing based on model output;
- tool dispatch;
- stopping;
- budgets;
- approval boundaries.

A framework is useful when it removes repetitive orchestration while keeping important application policies visible.


## Map familiar concepts to LangGraph

| First principles | LangGraph |
|---|---|
| state dataclass/dict | graph state schema |
| `while` / `for` loop | graph edges that revisit nodes |
| model call | model node |
| tool dispatcher | tool node |
| `if tool_call ...` | conditional edge/router |
| return final result | route to `END` |
| persisted checkpoint | checkpointer/persistence capability |
| approval pause | interrupt/pause-resume capability |

The names change; the underlying engineering concerns do not.


## Inspect the graph implementation

The runnable implementation lives in `framework_agent.py`. It uses a small `TypedDict` state, one model node, one tool node, and a router.


In [ ]:
from framework_agent import build_graph, initial_state


## The graph replaces the handwritten control loop

Conceptually, the graph is:

```text
START
  ↓
model
  ├── final / stopped → END
  └── tool request → tool
                      ↓
                    model
```

In Lesson 10 this routing lived in explicit loop code. Here, nodes and edges describe the same transition structure.


## Build and run it

This uses the same direct OpenAI SDK style as earlier lessons. LangGraph orchestrates node transitions; it does not replace the model provider API.


In [ ]:
graph = build_graph(max_steps=5)
state = initial_state(
    "Suggest one family-friendly activity in Melbourne. Use the activity tool before answering."
)
result = graph.invoke(state)
print("status:", result["status"])
print("steps:", result["steps"])
print("tool calls:", result["tool_calls"])
print("answer:", result["answer"])


## What got simpler?

The runner no longer contains the explicit `while True`, node-selection logic, and transition plumbing. The graph definition makes possible transitions visible as topology.

That becomes increasingly valuable when systems have multiple states such as planning, execution, approval, fallback, escalation, and repair.


## What did **not** disappear?

The tool node still validates tool name and arguments. We still decide what state is authoritative. We still need limits. We still need permissions, retry policy, observability, and user-safe failure behavior.

A graph framework orchestrates application logic; it does not define the application's trust boundaries for us.


## First-principles loop vs graph

| Concern | Handwritten runner | LangGraph |
|---|---|---|
| control flow | explicit Python loop | graph nodes/edges |
| state transitions | mutate/update state yourself | nodes return state updates |
| branching | Python `if`/`match` | conditional edges |
| loops | explicit iteration | graph cycles |
| visual topology | must infer from code | graph structure is explicit |
| persistence | implement yourself | framework checkpointer support |
| pause/resume | implement yourself | framework interrupt/checkpoint primitives |
| tool security | still yours | still yours |
| domain validation | still yours | still yours |
| cost/retry policy | still yours | still yours |


## Why not start the course here?

Without Lessons 2–10, `StateGraph`, nodes, edges, routers, persistence, and interrupts can look like framework-specific magic. Now each abstraction maps to a mechanism we already built.

This is the reason the course introduced frameworks late rather than early.


## Framework abstraction can leak

Real systems still care about provider-specific behavior: tool-call formats, streaming, structured outputs, token accounting, errors, reasoning modes, caching, and multimodal inputs. Orchestration frameworks reduce integration work but cannot erase all provider differences.


## Human-in-the-loop and persistence

LangGraph's larger value appears when a run must survive process boundaries: persist state, pause at an approval node, and resume later. Those capabilities correspond directly to the checkpoint/resume architecture we discussed in Lesson 8.

For a tiny synchronous agent, a graph can be more abstraction than you need. For long-running stateful workflows, the trade-off becomes much more attractive.


## When would I use which approach?

**Plain Python:** small agent, few transitions, maximum transparency, teaching/prototyping, or very custom control flow.

**LangGraph-style orchestration:** multiple states, cycles, durable execution, explicit branching, checkpointing, pause/resume, streaming, or growing operational complexity.

**Higher-level prebuilt agent:** standard agent architecture where development speed matters more than controlling every orchestration detail.

The more abstraction you adopt, the more important it is to know which behavior the framework owns and where you still need application policy.


## Exercises

1. Add a second read-only tool and route it through the same tool node.
2. Add a deterministic tool-call budget to graph state and stop when it is exhausted.
3. Draw how the Lesson 10 `WAITING_FOR_APPROVAL` state would become an approval node plus persisted checkpoint.
4. Compare a plain Python trace and LangGraph trace for the same goal. Which is easier to debug?
5. Identify which Lesson 10 code a framework removes and which code should **not** be removed because it represents policy.
6. Consider an incident-response agent with investigate → diagnose → approval → remediation → verify nodes. Decide where cycles are useful and where deterministic edges are safer.


## Module 03 checkpoint

You should now be able to look at an agent framework and recognize the underlying pieces: tools, state, control loops, routing, planning, retries, budgets, approvals, and termination.

The next module moves from a single agent toward **agentic systems**: larger architectures, coordination patterns, task decomposition, multiple specialized components, and the operational boundaries around them.
